# Getting started with psp.clobe

*The vignette `vignette("clobe")` of the R package clobe, ported to `psp.clobe`.*

clobe compares the *range of behaviour* a computational model can produce with the range of behaviour people actually show. Instead of asking "how close is the model to the average participant?", it asks two questions:

1. Can the model produce what each participant did? (**accommodation**)
2. Does the model also produce many things nobody did? (**excess flexibility**)

g-distance (Dome & Wills, 2025) combines the two into a single number. This notebook walks through the whole workflow on simulated data, and then does the same with a parameter space partitioning run in place of a grid. The other notebooks explain each step and its mathematics in more detail:

- [inequality-matrices.ipynb](inequality-matrices.ipynb): how a result becomes an ordinal pattern.
- [universal-set.ipynb](universal-set.ipynb): counting every pattern that could occur.
- [gdistance.ipynb](gdistance.ipynb): accommodation, excess flexibility and g-distance.

In [1]:
import pandas as pd
import torch
from IPython.display import Markdown, display

from psp import clobe

torch.set_printoptions(precision=2)

## The experiment

Imagine an experiment with four conditions (we will call them *stimuli*). Each of 40 participants does 20 trials per stimulus, and we record the proportion of trials on which they gave a particular response. Participants differ: some respond more often as we go from stimulus 1 to stimulus 4, some less often, and some barely change.

The data are simulated as in the R vignette, with torch's random number generator, so the numbers differ from the R version.

In [2]:
g = torch.Generator().manual_seed(2025)
x = torch.tensor([-1.5, -0.5, 0.5, 1.5], dtype=torch.float64)  # a feature of each stimulus
slopes = 0.5 + torch.randn(40, generator=g, dtype=torch.float64)
p = torch.sigmoid(slopes[:, None] * x + 0.3 * torch.randn(40, 4, generator=g, dtype=torch.float64))
human = torch.binomial(torch.full_like(p, 20.0), p, generator=g) / 20
human[:6]

tensor([[0.30, 0.25, 0.65, 0.65],
        [0.65, 0.80, 0.20, 0.35],
        [0.55, 0.65, 0.40, 0.50],
        [0.55, 0.30, 0.70, 0.40],
        [0.30, 0.50, 0.45, 0.55],
        [0.10, 0.30, 0.75, 0.85]], dtype=torch.float64)

## Step 1: turn each result into a pattern

Averaging these rows would hide the differences between participants. Instead, we describe each participant by the *ordinal pattern* of their results: for every pair of stimuli, is the first larger, smaller, or about the same as the second? "About the same" means the difference is no larger than a threshold, here 0.1 (two trials out of twenty).

In [3]:
thresholds = (0.1, -0.1)
clobe.imac(human[0], thresholds)

tensor([[ 2.,  0., -1., -1.],
        [ 2.,  2., -1., -1.],
        [ 2.,  2.,  2.,  0.],
        [ 2.,  2.,  2.,  2.]], dtype=torch.float64)

Read the matrix row against column: the entry in row 0 and column 2 says how stimulus 0 compares with stimulus 2 (1 larger, -1 smaller, 0 about the same; indices start at 0, as everywhere in Python). Only the cells above the diagonal carry information; the others are set to 2.

`patterns()` does this for every participant and keeps each distinct pattern once, with how often it occurred:

In [4]:
people = clobe.patterns(human, thresholds)
people.n, people.counts

(22,
 tensor([3, 1, 1, 1, 4, 1, 2, 1, 2, 1, 9, 3, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1]))

In [5]:
display(Markdown(f"The {people.n} distinct patterns among 40 participants are the behaviour the model should be able to produce."))

The 22 distinct patterns among 40 participants are the behaviour the model should be able to produce.

## Step 2: what could anyone have done?

To judge whether a model is too flexible, we need to know how many patterns were possible at all: the **universal set**. With four stimuli and "about the same" meaning "within the threshold", the patterns are what mathematicians call *semiorders*, and there are exactly

In [6]:
clobe.semiorders(4)

183

of them. Because our measurements are proportions of 20 trials, we can also check that this measurement scale can produce every one of them:

In [7]:
clobe.brutes(torch.arange(21, dtype=torch.float64) / 20, 4, thresholds, return_matrices=False).n

183

It can, so the universal set has `semiorders(4)` patterns.

## Step 3: what can the model do?

A model is a set of patterns too: the patterns it produces across its parameter space. Here are two toy models of the response probabilities. The first can only produce monotonic changes across the stimuli; the second adds a quadratic term, so it can also produce U-shapes and inverted U-shapes. Both are written for a whole batch of parameter sets at once, as `psp` models are.

In [8]:
def linear(b):
    return torch.sigmoid(b[:, None] * x)


def quadratic(theta):
    return torch.sigmoid(theta[:, :1] * x + theta[:, 1:] * x**2)


grid = torch.arange(-30, 31, dtype=torch.float64) / 10
b, c = torch.meshgrid(grid, grid, indexing="ij")
models = {
    "linear": clobe.patterns(linear(grid), thresholds),
    "quadratic": clobe.patterns(quadratic(torch.stack([b.reshape(-1), c.reshape(-1)], 1)), thresholds),
}
{name: model.n for name, model in models.items()}

{'linear': 7, 'quadratic': 73}

## Step 4: compare

`gdistance()` takes the human patterns, the model patterns, the size of the universal set, a weight, and how often each human pattern occurred.

In [9]:
def compare(model):
    out = clobe.gdistance(people.inequality_matrices, model.inequality_matrices,
                          universal=clobe.semiorders(4), weight=0.5,
                          frequencies=people.frequencies)
    return {"alpha": out.alpha, "beta": out.beta, "g": out.gdistance}


results = pd.DataFrame({name: compare(model) for name, model in models.items()}).T
results.round(3)

,alpha,beta,g
linear,0.225,0.037,0.549
quadratic,0.575,0.385,0.406


- **alpha** is the proportion of participants whose pattern the model can produce. Higher is better.
- **beta** is the proportion of the patterns *nobody showed* that the model produces anyway. Lower is better.
- **g** is the distance from the ideal model, which produces every observed pattern and nothing else. Lower is better; it runs from 0 to 1.

In [10]:
r = results.round(3)
display(Markdown(
    f"The quadratic model accommodates {'more' if r.alpha.quadratic > r.alpha.linear else 'no more'} participants "
    f"({r.alpha.quadratic} against {r.alpha.linear}), and produces patterns that nobody produced "
    f"(beta of {r.beta.quadratic} against {r.beta.linear}). g-distance weighs the two; with `weight=0.5` they "
    f"count equally, and here the {r.g.idxmin()} model ends up closer to the ideal. "
    "[gdistance.ipynb](gdistance.ipynb) explains the formula and how to choose the weight."
))

The quadratic model accommodates more participants (0.575 against 0.225), and produces patterns that nobody produced (beta of 0.385 against 0.037). g-distance weighs the two; with `weight=0.5` they count equally, and here the quadratic model ends up closer to the ideal. [gdistance.ipynb](gdistance.ipynb) explains the formula and how to choose the weight.

## Step 3 again, with parameter space partitioning

A grid is fine for two parameters, but it is not how one finds the patterns of a model with many: that is the job of `psp.PSP`, which searches the parameter space for every pattern the model can produce. `clobe.imac` works on a whole batch of results, so it is the `discretize` of the run as it is, and the patterns the run finds go straight into `gdistance()`.

In [11]:
from functools import partial

from psp import PSP

run = PSP(
    quadratic,
    partial(clobe.imac, thresholds=thresholds),
    {"lower": [-3, -3], "upper": [3, 3], "init": [[0.0, 0.0]], "radius": 0.5,
     "population": 200, "iterations": 300},
    generator=torch.Generator().manual_seed(1),
    quiet=True,
)
found = clobe.gdistance(people.inequality_matrices, run.ordinal_patterns,
                        universal=clobe.semiorders(4), weight=0.5, frequencies=people.frequencies)


def keys(matrices):
    i, j = torch.triu_indices(4, 4, offset=1)
    return {tuple(row.tolist()) for row in matrices[:, i, j]}


by_search, by_grid = keys(run.ordinal_patterns), keys(models["quadratic"].inequality_matrices)
grid_g = results.g.quadratic
display(Markdown(
    f"The search finds {len(by_search)} patterns and the grid {len(by_grid)}: {len(by_search & by_grid)} "
    f"are found by both, {len(by_search - by_grid)} only by the search and {len(by_grid - by_search)} only "
    "by the grid. Neither is complete: a region of the parameter space that lies between the points of "
    "the grid is invisible to the grid, however fine it looks, and the search stops after 300 iterations, "
    f"before it has reached every region. With its patterns, the quadratic model has α = {found.alpha:.3f}, β = {found.beta:.3f} and "
    f"g = {found.gdistance:.3f}, against g = {grid_g:.3f} from the grid: the model is judged on the "
    "patterns it can produce, so how they are found matters."
))

The search finds 76 patterns and the grid 73: 70 are found by both, 6 only by the search and 3 only by the grid. Neither is complete: a region of the parameter space that lies between the points of the grid is invisible to the grid, however fine it looks, and the search stops after 300 iterations, before it has reached every region. With its patterns, the quadratic model has α = 0.700, β = 0.391 and g = 0.349, against g = 0.406 from the grid: the model is judged on the patterns it can produce, so how they are found matters.

## Reference

Dome, L., & Wills, A. J. (2025). g-Distance: On the comparison of model and human heterogeneity. *Psychological Review, 132*(3), 632-655. https://doi.org/10.1037/rev0000550